# 1. Notebook Overview

**Purpose:**  
This notebook performs **pRF sampling of precomputed steerable-pyramid feature maps**.

---

## Dependencies (Must Run First)

Before running this notebook, you **must complete the previous step**:

- `0_nsd_steerable_pipeline.ipynb`  
  → Computes and saves steerable pyramid feature maps for all images

This notebook **assumes these feature maps already exist** and will fail without them.

---

## Output Usage (Next Step)

The outputs generated in this notebook are used in the next stage:

- `2_nsd_regresion_prf_split.ipynb`  
  → Performs voxel-wise regression using the pRF-sampled feature representations

This notebook produces the feature inputs required for that regression step.

---

##  What This Notebook Does

For each voxel:
- The population receptive field (pRF) is modeled as a **2D Gaussian**
- Each feature map is **spatially pooled** by computing the dot product between the Gaussian pRF and the feature map

The output is an **HDF5 file** containing voxel-specific feature values, which are later used for:
- session-wise regression  
- representational drift analysis  

---


# 2. Required Inputs and Data Structure

All necessary files are stored in the lab Google Drive:

**Path:**  
`My Drive → V1_Drift`

### Main folders:

1. **`prfsample_expand/`**  
   - Output of this notebook (pRF-sampled features per subject)

2. **`NSD/pyramid_expand/`**  
   - Precomputed steerable pyramid outputs  
   - Contains feature maps for ~10,000 images per subject  
   - Note: Due to storage limits, processing is done **per subject**, and files are replaced as needed

3. **`NSD/<subject_folder>/`**  
   - Contains subject-specific data:
     - fMRI beta responses
     - pRF parameters
     - ROI definitions

4. **`NSD/nsddata/experiments/nsd/`**  
   - Experimental design files (image ordering, metadata)

---

# 3. Runtime and Processing Details

- Processing is performed **per subject**
- Typical configuration:
  - **10 CPU cores**
  - **Batch size: 10 images**
- For some subjects (e.g., ~40 sessions), processing includes **~10,000 images**

Due to the large dataset size (~72k images total, several TB), computation and storage are handled incrementally.


---

# 4. Using This Notebook with Non-NSD Data

This notebook is not limited to the NSD dataset and can be applied to any dataset, provided the required inputs are supplied.

To run this pipeline on new data, you must provide:

### 1. Image List
- A list of image IDs (or filenames) to process  
- These must match the naming of the precomputed feature files

### 2. Steerable Pyramid Features
- A folder containing precomputed steerable pyramid outputs
- Files must follow a **consistent naming convention** (e.g., one file per image)
- Each file should contain:
  - Multi-scale, multi-orientation feature maps (e.g., `sumOri`, `modelOri`)

### 3. Voxel-wise pRF Parameters

For each voxel, you must provide:

- **pRF eccentricity**
- **pRF polar angle**
- **pRF size** (standard deviation of Gaussian)
- **ROI label** (to group voxels by region)
- *(optional)* **pRF R²** (quality metric for filtering voxels)

These parameters define the 2D Gaussian used for spatial pooling.

### 4. ROI Mapping

- A mapping from ROI labels to visual regions (e.g., V1, V2, V3, hV4)
- This is used to group voxels and organize outputs

---

### Important Notes

- pRFs are assumed to be defined in **visual degrees** and converted internally to pixel space
- All feature maps must have the **same spatial resolution** (e.g., 256×256)
- The pipeline assumes **one pRF per voxel** and **fixed pRFs across sessions**

# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Inspect a Random Steerable Pyramid File (Sanity Check)


In [ ]:
import os, numpy as np, random, matplotlib.pyplot as plt

# --- Config ---
folder_path = "/content/drive/MyDrive/V1_Drift/NSD/pyramid_expand"
SHOW_IMAGE = True   # set False if you don’t want to visualize any maps

# --- Step 1: pick a random .npz file ---
files = [f for f in os.listdir(folder_path) if f.endswith(".npz")]
if not files:
    raise FileNotFoundError(f"No .npz files found in {folder_path}")

rand_file = random.choice(files)
file_path = os.path.join(folder_path, rand_file)
print(f"\n Checking random pyramid file:\n{file_path}")

# --- Step 2: load file safely ---
try:
    data = np.load(file_path, allow_pickle=True)
except Exception as e:
    raise IOError(f"Error reading {file_path}: {e}")

# --- Step 3: inspect keys ---
print("\nAvailable keys:")
for k in data.keys():
    print(" •", k)

# --- Step 4: inspect each entry ---
for key in data.files:
    arr = data[key]
    if isinstance(arr, np.ndarray):
        print(f"\nKey: {key}")
        if arr.dtype == "object":
            print(f"  type: object array (probably list of arrays), len={len(arr)}")
            # Inspect first element
            first = arr[0]
            if isinstance(first, np.ndarray):
                print(f"  first element shape: {first.shape}, dtype: {first.dtype}")
                print(f"  mean={np.mean(first):.4f}, std={np.std(first):.4f}, "
                      f"min={np.min(first):.4f}, max={np.max(first):.4f}")
        else:
            print(f"  shape={arr.shape}, dtype={arr.dtype}")
            print(f"  mean={np.mean(arr):.4f}, std={np.std(arr):.4f}, "
                  f"min={np.min(arr):.4f}, max={np.max(arr):.4f}")
    else:
        print(f"\nKey: {key} (non-array type: {type(arr)})")

# --- Step 5: optional visualization ---
if SHOW_IMAGE:
    if "sumOri" in data:
        sumOri = data["sumOri"]
        if isinstance(sumOri, np.ndarray) and len(sumOri) > 0:
            first_band = sumOri[0]
            plt.imshow(first_band, cmap="gray")
            plt.title("sumOri[0]")
            plt.colorbar()
            plt.show()

    if "modelOri" in data:
        modelOri = data["modelOri"]
        if isinstance(modelOri, np.ndarray) and len(modelOri) > 0:
            first_set = modelOri[0]
            if isinstance(first_set, list) and len(first_set) > 0:
                first_ori = np.array(first_set[0])
                plt.imshow(first_ori, cmap="inferno")
                plt.title("modelOri[0][0]")
                plt.colorbar()
                plt.show()

data.close()


# checks that all the images the subject saw are present in the folder. and if not print which isnt

In [ ]:
import os
import re
import scipy.io as sio
import numpy as np





# ----------------------------------
# Configuration
# ----------------------------------
subject_index = 4  # 1-based
pyramid_folder = "/content/drive/MyDrive/V1_Drift/pyramid_expand"
expdesign_path = "/content/drive/MyDrive/V1_Drift/NSD/nsddata/experiments/nsd/nsd_expdesign.mat"



# ----------------------------------
# Restore files from trash_not_subject back to pyramid folder
# ----------------------------------
trash_not_subject = os.path.join(pyramid_folder, "trash_not_subject")

if os.path.isdir(trash_not_subject):
    restored = 0
    skipped = 0

    for f in os.listdir(trash_not_subject):
        if not (f.startswith("pyrImg") and f.endswith(".npz")):
            continue

        src = os.path.join(trash_not_subject, f)
        dst = os.path.join(pyramid_folder, f)

        if os.path.exists(dst):
            # Do NOT overwrite existing files
            skipped += 1
            continue

        os.rename(src, dst)
        restored += 1

    print("---------------------------------------------------")
    print(f"Restored {restored} files from trash_not_subject to pyramid folder.")
    if skipped > 0:
        print(f"Skipped {skipped} files because they already exist in pyramid folder.")
else:
    print("---------------------------------------------------")
    print("No trash_not_subject folder found. Nothing to restore.")



# ----------------------------------
# Load subject images from design file
# ----------------------------------
mat_data = sio.loadmat(expdesign_path)
subjectim = mat_data["subjectim"]
masterordering = mat_data["masterordering"].flatten() - 1  # convert to 0-based

valid_trials = masterordering[masterordering < subjectim.shape[1]]
allImgs = np.unique(subjectim[subject_index - 1, valid_trials])  # NSD 1-based IDs
allImgs_set = set(int(x) for x in allImgs)

print(f"Subject {subject_index} saw {len(allImgs_set)} unique images.")

# ----------------------------------
# Scan pyramid folder and group files by image index
# ----------------------------------
img_to_files = {}  # key: zero-based img index, value: list of filenames

for f in os.listdir(pyramid_folder):
    full = os.path.join(pyramid_folder, f)
    if os.path.isdir(full):
        continue
    if not (f.startswith("pyrImg") and f.endswith(".npz")):
        continue

    # parse digits after "pyrImg" (ignore " (1)" etc.)
    m = re.match(r"pyrImg(\d+)", f)
    if not m:
        print(f"Warning: Could not parse file name: {f}")
        continue

    img0 = int(m.group(1))  # zero-based ID in filename
    img_to_files.setdefault(img0, []).append(f)

# ----------------------------------
# Handle duplicates:
# Keep one canonical file pyrImg{idx}.npz in pyramid_folder.
# DELETE all other duplicates (no trash).
# ----------------------------------
dup_deleted = 0
dup_groups = 0

for img0, flist in list(img_to_files.items()):
    if len(flist) <= 1:
        continue

    dup_groups += 1
    canonical = f"pyrImg{img0}.npz"

    # Ensure canonical exists in root
    if canonical in flist:
        keep = canonical
    else:
        # rename first file to canonical
        keep = flist[0]
        src = os.path.join(pyramid_folder, keep)
        dst = os.path.join(pyramid_folder, canonical)
        if src != dst:
            os.rename(src, dst)
            print(f"Renamed {keep} -> {canonical}")
        keep = canonical

    # Delete all other duplicates
    for f in flist:
        if f == keep:
            continue
        fp = os.path.join(pyramid_folder, f)
        if os.path.exists(fp):
            os.remove(fp)
            dup_deleted += 1

    # keep dict clean
    img_to_files[img0] = [keep]

print(f"Duplicate handling: {dup_groups} images had duplicates; deleted {dup_deleted} duplicate files.")

# ----------------------------------
# Build existing image set (NSD 1-based IDs)
# ----------------------------------
existing_set = set(int(img0) + 1 for img0 in img_to_files.keys())

# ----------------------------------
# Compare sets: missing vs extra
# Extra files remain in pyramid_folder (we DO NOT move them).
# ----------------------------------
missing_imgs = sorted(allImgs_set - existing_set)
extra_imgs = sorted(existing_set - allImgs_set)

print("---------------------------------------------------")
if missing_imgs:
    print(f"Missing {len(missing_imgs)} pyramid files for this subject.")
    print("First 20 missing:", missing_imgs[:20], "..." if len(missing_imgs) > 20 else "")
else:
    print("No missing pyramid files for this subject.")

print("---------------------------------------------------")
if extra_imgs:
    print(f"Found {len(extra_imgs)} EXTRA pyramid files (not seen by subject).")
    print("First 20 extra:", extra_imgs[:20], "..." if len(extra_imgs) > 20 else "")
    print("NOTE: Extras were NOT moved; they remain in the pyramid folder.")
else:
    print("No extra pyramid files.")

print("Done.")




# The prf sampling main code

In [ ]:
import os
import time
import numpy as np
import nibabel as nib  # Used to load .nii.gz neuroimaging files
import scipy.io as sio  # For loading MATLAB .mat files
import h5py  # For working with HDF5 files
import gc  # For manual garbage collection to free memory
from joblib import Parallel, delayed  # For parallel processing
from tqdm import tqdm  # for progress bars
import tempfile, shutil  # for safely saving output files
import psutil  # RAM/Disk guards & monitors

# mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ===============================
# Main pRF Sampling Pipeline
# ===============================
# This code samples precomputed steerable pyramid feature maps
# using voxel-specific pRF Gaussian masks.
#
# Input:
#   - Precomputed steerable pyramid .npz files
#   - Subject-specific pRF parameter maps
#   - ROI label maps
#   - NSD experimental design file
#
# Output:
#   - HDF5 file containing pRF-sampled feature values:
#       prfSampleLev
#       prfSampleLevOri
#       prfParam
#       voxel_ids
#       allImgs
#       trial_imgids
#
# Used in next step:
#   2. nsd_regresion_prf_split.ipynb
# ============================================================


# -----------------------------------------------------------------------
# Global toggles (for speed, keep these False)
# -----------------------------------------------------------------------
VERBOSE = False          # If True, show extra debug / sanity info
MONITOR_USAGE = False    # If True, print RAM/disk usage per batch

# -----------------------------------------------------------------------
# Paths config
# -----------------------------------------------------------------------
nsd_folder = "/content/drive/MyDrive/V1_Drift/NSD"  # Path to NSD dataset
pyramid_folder = "/content/drive/MyDrive/V1_Drift/pyramid_expand"  # Folder with steerable pyramid outputs
prf_folder = "/content/drive/MyDrive/V1_Drift/prfsample_expand"  # Folder to save output of pRF sampling

# -----------------------------------------------------------------------
# General settings
# -----------------------------------------------------------------------
interp_img_size = 256  # Interpolated image size used in model
background_size = 0  # Size of the padded image canvas
img_scaling = 1.0  # Image scaling factor
num_levels = 4  # Number of pyramid spatial frequency levels
num_orientations = 4  # Number of orientation filters
pix_per_deg = interp_img_size / 8.4  # pixels per degree
deg_per_pix = 8.4 / interp_img_size  # degrees per pixel

roi_names = ["V1v", "V1d", "V2v", "V2d", "V3v", "V3d", "hV4"]
test_mode = False  # Toggle test mode (fewer images) for debugging

# ------------------------------------------------------------
# Limit BLAS threads globally
# ------------------------------------------------------------
# NumPy and SciPy sometimes use multithreaded linear algebra internally.
# In this notebook, we also use joblib for parallel image processing.
#
# If each joblib worker also starts many internal BLAS threads, the runtime
# can become slower or unstable because too many CPU threads are created.
#
# Setting these environment variables to "1" limits each worker to one
# internal numerical thread.
#
# Keep this if:
#   - using joblib Parallel
#   - running many images
#   - running in Colab or another shared environment
#
# It can be removed if:
#   - running fully serial code
#   - not using joblib
#   - you intentionally want NumPy/SciPy to use multithreading
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# -----------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------


# Optional RAM and disk usage monitor

def show_usage(tag=""):
      """
    Print current RAM and disk usage.

    This is only for monitoring/debugging. It does not affect the analysis.

    Useful when:
        - running many batches
        - debugging Colab crashes
        - checking whether local disk/cache is filling up

    Can be removed if:
        - the dataset is small
        - memory and disk usage are not a concern
    """
    if not MONITOR_USAGE:
        return
    ram_gb = psutil.Process().memory_info().rss / 1024**3
    total, used, free = shutil.disk_usage("/content")
    print(
        f"[{tag}] RAM {ram_gb:5.2f} GB | "
        f"Disk used {used/1e9:6.1f}/{total/1e9:6.1f} GB | "
        f"Free {free/1e9:6.1f} GB"
    )

# Optional local cache size control

def enforce_cache_budget(cache_path="/content/pyr_cache",
                         max_bytes=3_000_000_000,
                         exts=(".npy", ".npz", ".pkl"),
                         verbose=False):
        """
    Keep the local cache folder below a defined disk-size limit.

    The pipeline may cache decompressed pyramid files locally to speed up
    repeated access. On Colab, local disk space is limited, so this function
    deletes the oldest cached files when the cache exceeds `max_bytes`.

    This does not delete the original pyramid files on Google Drive.
    It only deletes temporary local cache files.

    Useful when:
        - processing many images
        - repeatedly loading large .npz files
        - Colab local disk is filling up

    Can be removed if:
        - caching is disabled
        - the dataset is small
        - there is enough local disk space
    """
    if not os.path.isdir(cache_path):
        return
    files = []
    for root, _, names in os.walk(cache_path):
        for n in names:
            if n.endswith(exts):
                fp = os.path.join(root, n)
                try:
                    files.append((fp, os.path.getmtime(fp), os.path.getsize(fp)))
                except FileNotFoundError:
                    pass
    total = sum(sz for _, _, sz in files)
    if total <= max_bytes:
        return
    files.sort(key=lambda t: t[1])  # oldest first
    if verbose:
        print(
            f"[CACHE] Over budget: {total/1e9:.2f} GB > "
            f"{max_bytes/1e9:.2f} GB. Trimming..."
        )
    for fp, _, sz in files:
        try:
            os.remove(fp)
            total -= sz
            if verbose:
                print(
                    f"  deleted {os.path.basename(fp)} "
                    f"({sz/1e6:.1f} MB) -> {total/1e9:.2f} GB"
                )
            if total <= max_bytes:
                break
        except FileNotFoundError:
            continue

# Optional RAM pressure guard

def enforce_ram_budget(min_free_gb=2.0, on_pressure=None, tag=""):
        """
    Run garbage collection when available RAM is low.

    This function checks available system RAM. If free RAM drops below
    `min_free_gb`, it triggers Python garbage collection. An optional callback
    can also be used to clear custom in-memory caches.

    Useful when:
        - Colab crashes during long batch processing
        - many images or voxels are processed in parallel
        - large arrays are temporarily created inside each batch

    Can be removed if:
        - memory usage is stable
        - processing fewer images
        - running on a machine with enough RAM
    """
    avail_gb = psutil.virtual_memory().available / 1024**3
    if avail_gb < min_free_gb:
        if VERBOSE:
            print(
                f"[RAM] Low free RAM ({avail_gb:.2f} GB). "
                f"Triggering GC{' + callback' if on_pressure else ''}... ({tag})"
            )
        gc.collect()
        if on_pressure:
            on_pressure()


# Share pRF Gaussian masks with worker processes

def set_globals(roiGaus_shared):
    """
    Store ROI Gaussian masks in a global variable for worker functions.

    The image-processing function `process_single_image` needs access to
    the pRF Gaussian masks for each ROI. This helper places the Gaussian
    masks in a global variable before joblib launches parallel workers.

    This is part of the implementation of the parallel version.

    Keep this if:
        - using joblib Parallel
        - `process_single_image` reads `GLOBAL_roiGaus`

    Can be removed only if:
        - the code is rewritten so Gaussian masks are passed explicitly
          to each worker
        - the pipeline is rewritten to run serially
    """
    global GLOBAL_roiGaus
    GLOBAL_roiGaus = roiGaus_shared


# Load NIfTI files safely
def load_nifti(filepath):
    """
    Load a NIfTI neuroimaging file if it exists.

    Returns:
        - NIfTI data as a NumPy array if the file exists
        - None if the file is missing

    This is used for loading:
        - pRF angle maps
        - pRF eccentricity maps
        - pRF size maps
        - pRF R2 maps
        - visual ROI label maps

    Keep this if your pRF and ROI data are stored as .nii or .nii.gz files.

    Replace this if:
        - the new dataset stores pRF/ROI information in another format
          such as .csv, .npy, .mat, or HDF5
    """
    return nib.load(filepath).get_fdata() if os.path.exists(filepath) else None


# Limit BLAS threads inside each worker process
_LIMIT_THREADS_DONE = False

def _limit_threads_once():
    """
    Apply the one-thread BLAS limit inside a joblib worker process.

    The global environment variables above are set in the main process.
    This helper repeats the same protection inside worker processes.

    Useful when:
        - using joblib with multiple processes
        - NumPy operations are called inside each worker

    Can be removed if:
        - not using joblib
        - processing images serially
    """
    global _LIMIT_THREADS_DONE
    if _LIMIT_THREADS_DONE:
        return
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
    _LIMIT_THREADS_DONE = True

# ------------------------------------------------------------
# Local cache directory
# ------------------------------------------------------------
# This folder is used to store temporary local copies of loaded pyramid data.
# Reading repeatedly from Google Drive can be slow. Caching locally can speed up
# the pipeline, especially when the same files are accessed more than once.
#
# For smaller datasets, this cache may be optional.cache_dir = "/content/pyr_cache"
os.makedirs(cache_dir, exist_ok=True)




def load_pyramid_data(imgNum):
    """
    Load steerable pyramid data (sumOri, modelOri) from .npz,
    and cache as .npy files for faster subsequent access.

    Args:
        imgNum (int): Image number (1-based)

    Returns:
        (sumOri, modelOri):
          sumOri: np.ndarray float32, shape (numLevels+2, H, W)
          modelOri: list, where modelOri[ilev] is typically (O,H,W) for ilev < num_levels
    """
    idx = imgNum - 1
    npz_path = os.path.join(pyramid_folder, f"pyrImg{idx}.npz")
    cached_sumOri = os.path.join(cache_dir, f"sumOri_{idx}.npy")
    cached_modelOri = os.path.join(cache_dir, f"modelOri_{idx}.npy")

    if os.path.exists(cached_sumOri) and os.path.exists(cached_modelOri):
        sumOri = np.load(cached_sumOri, allow_pickle=True).astype(np.float32, copy=False)
        modelOri = np.load(cached_modelOri, allow_pickle=True)
        # if cached as object array, convert to list for consistent indexing
        if isinstance(modelOri, np.ndarray) and modelOri.dtype == object:
            modelOri = list(modelOri)
        return sumOri, modelOri

    with np.load(npz_path, allow_pickle=True) as data:
        sumOri = data["sumOri"].astype(np.float32, copy=False)
        modelOri = data["modelOri"]

    # Normalize modelOri into a Python list
    # Handles cases where modelOri is saved either as:
    #   - an object array of levels
    #   - a one-element wrapper around a list
    if isinstance(modelOri, np.ndarray) and modelOri.shape == (1,) and isinstance(modelOri[0], list):
        modelOri = modelOri[0]
    elif isinstance(modelOri, np.ndarray) and modelOri.dtype == object:
        modelOri = list(modelOri)

    np.save(cached_sumOri, sumOri)
    np.save(cached_modelOri, np.array(modelOri, dtype=object))  # cache consistently as object

    return sumOri, modelOri


def process_single_image(imgNum, num_levels, num_orientations):
    _limit_threads_once()
    try:
        sumOri, modelOri = load_pyramid_data(imgNum)
    except Exception as e:
        if VERBOSE:
            print(f"[{imgNum}] load error: {e}")
        return imgNum, {}, {}

    prfSampleLev_partial = {}
    prfSampleLevOri_partial = {}

    L_sum = min(num_levels + 2, len(sumOri))
    L_ori = min(num_levels, len(modelOri))

    TARGET_L = num_levels + 2
    TARGET_O = num_orientations

    # Stack sums (L_sum, H, W)
    try:
        sum_stack = np.stack(sumOri[:L_sum], axis=0).astype(np.float32, copy=False)
        sum_stack = np.ascontiguousarray(sum_stack)
    except Exception as e:
        if VERBOSE:
            print(f"[{imgNum}] sum_stack error: {e}")
        return imgNum, {}, {}

    # Stack orientations (L_ori, O, H, W)
    try:
        ori_list = []
        for ilev in range(L_ori):
            om = modelOri[ilev]
            if isinstance(om, list):
                om = np.stack(om, axis=0)
            elif isinstance(om, np.ndarray) and om.ndim == 3:
                pass
            else:
                continue
            om = np.ascontiguousarray(om.astype(np.float32, copy=False))
            ori_list.append(om)
        if len(ori_list) > 0:
            max_O = max(arr.shape[0] for arr in ori_list)
            target_O_local = max(TARGET_O, max_O)
            fixed = []
            for arr in ori_list:
                if arr.shape[0] < target_O_local:
                    pad_O = target_O_local - arr.shape[0]
                    arr = np.pad(arr, ((0, pad_O), (0, 0), (0, 0)))
                fixed.append(arr)
            ori_stack = np.stack(fixed, axis=0).astype(np.float32, copy=False)
            ori_stack = np.ascontiguousarray(ori_stack)
        else:
            ori_stack = None
    except Exception as e:
        if VERBOSE:
            print(f"[{imgNum}] ori_stack error: {e}")
        return imgNum, {}, {}

    for roinum, G_all in GLOBAL_roiGaus.items():
        try:
            G_all = np.ascontiguousarray(G_all.astype(np.float32, copy=False))
            V, H, W = G_all.shape

            # ---------- Levels (keep +2 bands) ----------
            prf_levels = np.einsum("vhw,lhw->vl", G_all, sum_stack, optimize=True)
            # pad/trim to TARGET_L
            if prf_levels.shape[1] < TARGET_L:
                prf_levels = np.pad(
                    prf_levels, ((0, 0), (0, TARGET_L - prf_levels.shape[1]))
                )
            elif prf_levels.shape[1] > TARGET_L:
                prf_levels = prf_levels[:, :TARGET_L]
            prf_levels = prf_levels.astype(np.float32, copy=False)

            # ---------- Orientations (NO +2 here) ----------
            if ori_stack is not None and L_ori > 0:
                prf_oris = np.einsum("vhw,lohw->vlo", G_all, ori_stack, optimize=True)
                # trim/pad to exactly num_levels
                if prf_oris.shape[1] > num_levels:
                    prf_oris = prf_oris[:, :num_levels, :]
                elif prf_oris.shape[1] < num_levels:
                    prf_oris = np.pad(
                        prf_oris,
                        ((0, 0), (0, num_levels - prf_oris.shape[1]), (0, 0)),
                    )
                # pad orientations to num_orientations if needed
                if prf_oris.shape[2] < num_orientations:
                    prf_oris = np.pad(
                        prf_oris,
                        (
                            (0, 0),
                            (0, 0),
                            (0, num_orientations - prf_oris.shape[2]),
                        ),
                    )
            else:
                prf_oris = np.zeros(
                    (V, num_levels, num_orientations), dtype=np.float32
                )

            prf_oris = np.ascontiguousarray(prf_oris.astype(np.float32, copy=False))

            prfSampleLev_partial[roinum] = prf_levels
            prfSampleLevOri_partial[roinum] = prf_oris

        except Exception as e:
            if VERBOSE:
                print(f"[{imgNum}] ROI {roinum} einsum error: {e}")
            return imgNum, {}, {}

    try:
        del sum_stack
    except NameError:
        pass
    try:
        del ori_stack
    except NameError:
        pass
    gc.collect()

    return imgNum, prfSampleLev_partial, prfSampleLevOri_partial

def prf_sample_model_expand(isub, visualRegion, batch_size=100):
    """
    Main function to compute pRF samples by applying ROI Gaussians to steerable pyramid outputs.

    Args:
        isub: Subject index (1-based).
        visualRegion: Visual region group ID (1 to 4).
        batch_size: Number of images to process in parallel per batch.
    """
    print(f"\n-- Running pRF Sampling for Subject {isub}, Region {visualRegion} ---")

    # Output path on Drive (final)
    h5_filename_drive = os.path.join(
        prf_folder, f"prfSampleStim_v{visualRegion}_sub{isub}.h5"
    )

    # Local working file (write here during computation!)
    local_dir = "/content/work"
    os.makedirs(local_dir, exist_ok=True)
    h5_filename_local = os.path.join(local_dir, os.path.basename(h5_filename_drive))

    # We'll use the LOCAL file everywhere below
    h5_filename = h5_filename_local

    # Remove legacy .log file if exists
    log_path = os.path.join(prf_folder, f"processed_images_sub{isub}.log")
    if os.path.exists(log_path):
        os.remove(log_path)

    # Load NSD experimental design data
    nsdDesignPath = os.path.join(
        "/content/drive/MyDrive/V1_Drift/NSD/nsddata/experiments/nsd/nsd_expdesign.mat"
    )
    if not os.path.exists(nsdDesignPath):
        print(f"ERROR: NSD Design file not found: {nsdDesignPath}")
        return

    nsdDesign = sio.loadmat(nsdDesignPath)
    subjectim = nsdDesign["subjectim"]
    masterordering = nsdDesign["masterordering"].flatten() - 1
    valid_masterordering = masterordering[masterordering < subjectim.shape[1]]

    # trial-level (with repeats) and unique image IDs
    trial_imgids = subjectim[isub - 1, valid_masterordering].astype(np.int32)
    allImgs = np.unique(trial_imgids)

    if test_mode:
        allImgs = allImgs[:2000]

    print(f"Total unique images to consider: {len(allImgs)}")

    # Max number of processed images you expect to store in memmap log
    max_images = int(len(allImgs))
    log_mmap_path = os.path.join(local_dir, f"processed_images_sub{isub}_v{visualRegion}.mmap")


    # Initialize or load memory-mapped log file
    if os.path.exists(log_mmap_path):
        log_mmap = np.memmap(log_mmap_path, dtype="int32", mode="r+")
        current_log_index = np.count_nonzero(log_mmap)
        processed_images = set(int(x) for x in log_mmap[:current_log_index])
    else:
        log_mmap = np.memmap(log_mmap_path, dtype="int32", mode="w+", shape=(max_images,))
        log_mmap[:] = 0
        log_mmap.flush()
        current_log_index = 0
        processed_images = set()

    # ------------------------------------------------------------
    # Load pRF maps and ROI labels
    # ------------------------------------------------------------
    betas_folder = os.path.join(nsd_folder, f"subj{isub:02d}", "func1pt8mm")
    prf_files = [
        "prf_angle.nii.gz",
        "prf_eccentricity.nii.gz",
        "prf_size.nii.gz",
        "R2.nii.gz",
    ]

    angData, eccData, sizeData, r2Data = [
        load_nifti(os.path.join(betas_folder, f)) for f in prf_files
    ]
    visRoiData = load_nifti(os.path.join(betas_folder, "roi", "prf-visualrois.nii.gz"))

    if any(data is None for data in [angData, eccData, sizeData, r2Data, visRoiData]):
        print("ERROR: Missing pRF data. Check file paths.")
        return

    # ------------------------------------------------------------
    # Define ROIs per visualRegion
    # ------------------------------------------------------------
    roi_map = {
        1: [0, 1],  # V1v + V1d
        2: [2, 3],  # V2v + V2d
        3: [4, 5],  # V3v + V3d
        4: [6],     # hV4
    }
    rois = roi_map.get(visualRegion, [])

    # ------------------------------------------------------------
    # Setup output containers and load cached Gaussians if available
    # ------------------------------------------------------------
    gauss_file = os.path.join(prf_folder, f"roiGaussians_sub{isub}_v{visualRegion}.h5")
    roiGaus = {}
    roiPrf = {}
    missing_rois = []

    if os.path.exists(gauss_file):
        if VERBOSE:
            print(f"Loading cached Gaussians from {gauss_file}")
        with h5py.File(gauss_file, "r") as f:
            for roinum in rois:
                key = f"roi_{roinum}"
                if key in f:
                    G_all = f[key][:]
                    roiGaus[roinum] = G_all

                    roi_mask = visRoiData == (roinum + 1)
                    ecc_deg = eccData[roi_mask]
                    ang_deg = angData[roi_mask]
                    sz_deg = sizeData[roi_mask]
                    sz_pix = sz_deg * pix_per_deg

                    theta = np.deg2rad(ang_deg)
                    ecc_pix = ecc_deg * pix_per_deg

                    x0_pix = ecc_pix * np.cos(theta)
                    y0_pix = ecc_pix * np.sin(theta)

                    roiPrf[roinum] = {
                        "ecc": ecc_deg,
                        "ang": ang_deg,
                        "sz": sz_deg,
                        "x": x0_pix,
                        "y": y0_pix,
                    }
                    voxel_ids = np.flatnonzero(roi_mask.ravel()).astype(np.int32)
                    roiPrf[roinum]["voxel_ids"] = voxel_ids
                else:
                    missing_rois.append(roinum)
    else:
        missing_rois = rois

    # ------------------------------------------------------------
    # Compute missing Gaussian kernels
    # ------------------------------------------------------------
    if missing_rois:
        if VERBOSE:
            print(f"Computing Gaussians for missing ROIs: {missing_rois}")

        half = interp_img_size // 2   # 128.0 for 256
        coords = np.arange(-half, half, 1.0)  # [-128 .. 127], length 256
        X, Y = np.meshgrid(coords, -coords, indexing="xy")  # (256,256)

        with h5py.File(gauss_file, "a") as f:
            for roinum in missing_rois:
                roi_mask = visRoiData == (roinum + 1)
                if np.sum(roi_mask) == 0:
                    if VERBOSE:
                        print(f"Skipping empty ROI: {roinum}")
                    continue

                ecc_deg = eccData[roi_mask]
                ang_deg = angData[roi_mask]
                sz_deg  = sizeData[roi_mask]

                sz_pix = sz_deg * pix_per_deg
                sz_pix = np.maximum(sz_pix, 1e-6)  # guard

                theta   = np.deg2rad(ang_deg)
                ecc_pix = ecc_deg * pix_per_deg

                x0_pix = ecc_pix * np.cos(theta)
                y0_pix = ecc_pix * np.sin(theta)

                roiPrf[roinum] = {
                    "ecc": ecc_deg,
                    "ang": ang_deg,
                    "sz":  sz_deg,
                    "x":   x0_pix,
                    "y":   y0_pix,
                }

                X_diff = X[None, :, :] - x0_pix[:, None, None]
                Y_diff = Y[None, :, :] - y0_pix[:, None, None]

                G_all = np.exp(-(X_diff**2 + Y_diff**2) / (2.0 * (sz_pix[:, None, None] ** 2)))
                G_all = G_all.astype(np.float32, copy=False)

                roiGaus[roinum] = G_all

                key = f"roi_{roinum}"
                if key in f:
                    del f[key]  # avoid "name already exists" crash if rerunning
                f.create_dataset(key, data=G_all, compression="gzip")

        if VERBOSE:
            print("Updated Gaussian cache with missing ROIs")


    # ------------------------------------------------------------
    # Determine images to process (skip already processed; skip missing pyramid)
    # ------------------------------------------------------------
    work_list = []
    image_index_map = {}

    existing_all_imgs = None

    # Prefer resuming from LOCAL if it exists; otherwise, if Drive exists, copy it locally first
    if os.path.exists(h5_filename_local):
        src_for_resume = h5_filename_local
    elif os.path.exists(h5_filename_drive):
        print(f"[RESUME] Copying existing Drive H5 -> local: {h5_filename_drive} -> {h5_filename_local}")
        shutil.copy2(h5_filename_drive, h5_filename_local)
        src_for_resume = h5_filename_local
    else:
        src_for_resume = None

    if src_for_resume is not None:
        with h5py.File(src_for_resume, "r") as f:
            if "allImgs" in f:
                existing_all_imgs = f["allImgs"][:]

    # Canonical index map ALWAYS based on allImgs
    canon_idx = {int(img): i for i, img in enumerate(allImgs)}

    # If resuming, sanity check: stored allImgs must match canonical allImgs
    if existing_all_imgs is not None:
        if len(existing_all_imgs) != len(allImgs) or np.any(existing_all_imgs.astype(np.int32) != allImgs.astype(np.int32)):
            raise RuntimeError(
                "Existing H5 allImgs does not match current canonical allImgs. "
                "Delete the corrupted H5 (both local+Drive) and rerun."
            )

    for imgNum in allImgs:
        if int(imgNum) in processed_images:
            continue

        pyramid_filename = os.path.join(pyramid_folder, f"pyrImg{int(imgNum) - 1}.npz")
        if not os.path.exists(pyramid_filename):
            if VERBOSE:
                print(f"Missing pyramid file: {pyramid_filename}")
            continue

        work_list.append(int(imgNum))
        image_index_map[int(imgNum)] = canon_idx[int(imgNum)]

    if not work_list:
        print("\nNo new images to process. Skipping save.")
        return

    print(f"Processing {len(work_list)} new images")



    # ------------------------------------------------------------
    # Create / update HDF5 metadata + output datasets
    # ------------------------------------------------------------
    with h5py.File(h5_filename, "a") as f:
        if "allImgs" not in f:
            # IMPORTANT: allImgs must be the FULL canonical list
            f.create_dataset("allImgs", data=allImgs.astype(np.int32))


        if "trial_imgids" not in f:
            f.create_dataset(
                "trial_imgids",
                data=trial_imgids.astype(np.int32),
                compression="gzip",
            )

        if "rois" not in f:
            f.create_dataset("rois", data=np.array(rois, dtype=np.int32))

        f.attrs["numLevels"] = num_levels
        f.attrs["numOrientations"] = num_orientations
        f.attrs["interpImgSize"] = interp_img_size
        f.attrs["backgroundSize"] = background_size
        f.attrs["pixPerDeg"] = pix_per_deg
        f.attrs["roi_names"] = np.array(roi_names, dtype="S10")

        n_images = int(f["allImgs"].shape[0])

        # Create output datasets once
        for roinum, G_all in roiGaus.items():
            nvox = G_all.shape[0]
            dlev_key = f"prfSampleLev/roi_{roinum}"
            dori_key = f"prfSampleLevOri/roi_{roinum}"
            if dlev_key not in f:
                f.create_dataset(
                    dlev_key,
                    shape=(n_images, nvox, num_levels + 2),
                    maxshape=(n_images, nvox, num_levels + 2),
                    dtype=np.float32,
                    compression="gzip",
                )
            if dori_key not in f:
                f.create_dataset(
                    dori_key,
                    shape=(n_images, nvox, num_levels, num_orientations),
                    maxshape=(n_images, nvox, num_levels, num_orientations),
                    dtype=np.float32,
                    compression="gzip",
                )

        # voxel_ids per ROI
        if "voxel_ids" not in f:
            f.create_group("voxel_ids")
        for roinum in roiPrf:
            key_ids = f"voxel_ids/roi_{roinum}"
            if key_ids not in f:
                vid = roiPrf[roinum].get("voxel_ids")
                if vid is None:
                    roi_mask = visRoiData == (roinum + 1)
                    vid = np.flatnonzero(roi_mask.ravel()).astype(np.int32)
                f.create_dataset(
                    key_ids, data=vid, dtype=np.int32, compression="gzip"
                )

    # Pre-compute number of jobs once
    n_jobs = min(max(1, (os.cpu_count() or 2) - 1), 8)
    n_jobs = min(n_jobs, batch_size)

    num_batches = len(work_list) // batch_size + (
        1 if len(work_list) % batch_size != 0 else 0
    )

    start_time_global = time.time()
    MAX_RUNTIME_SECONDS = 23.5 * 3600  # Safety margin before 24h

    # ------------------------------------------------------------
    # Process images in batches
    # ------------------------------------------------------------
    for batch_num in range(num_batches):
        batch = work_list[batch_num * batch_size : (batch_num + 1) * batch_size]
        if not batch:
            continue
        if VERBOSE:
            print(
                f"Processing batch {batch_num + 1}/{num_batches} "
                f"with {len(batch)} images"
            )

        start = time.time()

        set_globals(roiGaus)

        results = Parallel(
            n_jobs=min(n_jobs, len(batch)),
            backend="loky",
            prefer="processes",
            batch_size="auto",
            verbose=0,
        )(
            delayed(process_single_image)(imgNum, num_levels, num_orientations)
            for imgNum in tqdm(
                batch,
                desc=f"Batch {batch_num + 1}/{num_batches}",
                unit="image",
                ncols=100,
                mininterval=0.5,
                disable=not VERBOSE,  # if you want *no* progress bar, set VERBOSE=False
            )
        )

        end = time.time()
        if VERBOSE:
            print(f"Batch {batch_num + 1} elapsed time: {end - start:.2f} s")

        # Write results directly to HDF5
        with h5py.File(h5_filename, "a") as f:
            for imgNum, partial_lev, partial_ori in results:
                if not partial_lev:
                    continue
                iimg = image_index_map[imgNum]
                for roinum in partial_lev:
                    f[f"prfSampleLev/roi_{roinum}"][iimg, :, :] = partial_lev[
                        roinum
                    ].astype(np.float32, copy=False)
                    f[f"prfSampleLevOri/roi_{roinum}"][iimg, :, :, :] = partial_ori[
                        roinum
                    ].astype(np.float32, copy=False)

        del results
        gc.collect()
        enforce_ram_budget(min_free_gb=2.0, tag=f"after batch {batch_num}")
        # Trim cache only once per batch (not per image)
        enforce_cache_budget(cache_dir, max_bytes=3_000_000_000, verbose=False)
        show_usage(f"batch {batch_num} saved")

        # Save pRF parameters once (idempotent)
        with h5py.File(h5_filename, "a") as f:
            if "prfParam" not in f:
                f.create_group("prfParam")

            for roinum in roiPrf:
                for param in ["ecc", "ang", "sz", "x", "y"]:
                    key = f"prfParam/{param}/roi_{roinum}"
                    data = roiPrf[roinum][param]
                    if key not in f:
                        f.create_dataset(key, data=data, compression="gzip")

            for roinum in roiPrf:
                key = f"prfParam/r2/roi_{roinum}"
                if key not in f:
                    roi_mask = visRoiData == (roinum + 1)
                    r2_roi = r2Data[roi_mask]
                    f.create_dataset(key, data=r2_roi, compression="gzip")

        # Update log for newly processed images
        new_logged_imgs = [img for img in batch if img not in processed_images]
        if new_logged_imgs:
            n = len(new_logged_imgs)
            if current_log_index + n > max_images:
                raise RuntimeError("Exceeded max_images capacity in memmap log.")
            log_mmap[current_log_index : current_log_index + n] = new_logged_imgs
            current_log_index += n
            log_mmap.flush()
            processed_images.update(new_logged_imgs)
            if VERBOSE:
                print(f"Logged {n} new images (mmap)")

        elapsed_global = time.time() - start_time_global
        if elapsed_global > MAX_RUNTIME_SECONDS:
            print(
                "Nearing Colab 24h timeout. Exiting early to save progress safely."
            )
            break

    # ------------------------------------------------------------
    # Finalize: copy local H5 -> Drive ONCE (atomic replace)
    # ------------------------------------------------------------
    try:
        # quick integrity check: can we open local file?
        with h5py.File(h5_filename_local, "r") as _f:
            _ = _f["allImgs"].shape[0]  # touch metadata

        tmp = h5_filename_drive + ".tmp"
        shutil.copy2(h5_filename_local, tmp)
        os.replace(tmp, h5_filename_drive)
        print(f"[DONE] Saved final H5 to Drive: {h5_filename_drive}")

    except Exception as e:
        print(f"[WARNING] Local H5 seems corrupted, NOT copying to Drive. Error: {e}")



# ------------------------------------------------------------
# Script entry point
# ------------------------------------------------------------
if __name__ == "__main__":
    prf_sample_model_expand(isub=1, visualRegion=1, batch_size=20)


# HDF5 Integrity Check: pRF Sampling Output

In [ ]:
import os, h5py, numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --------- config ----------
h5_path    = "/content/drive/MyDrive/V1_Drift/prfsample_expand/prfSampleStim_v4_sub4.h5"
roi_id     = 1
img_indices = [0, 1, 2]           # exactly 3 images (indices into allImgs)
vox_indices = [10, 200, 500]      # any set of voxels you want to overlay
SHOW_SUM_ORI_PER_LEVEL = False    # if True, dashed line shows Σ(orientations) per level
# ---------------------------

import os
import h5py
import numpy as np

def check_prf_hdf5_integrity(h5_path, verbose=True, check_values=True, atol=1e-7):
    """
    Validate structure, duplicates, corruption, and data consistency
    for prfSampleStim_vX_subY.h5 files.

    Args:
        h5_path (str): Path to the HDF5 file.
        verbose (bool): Print detailed results.
        check_values (bool): Check numeric sanity (mean/std, NaNs).
        atol (float): Tolerance for zero/NaN detection.
    """

    if not os.path.exists(h5_path):
        raise FileNotFoundError(f"File not found: {h5_path}")

    issues = []

    try:
        f = h5py.File(h5_path, "r")
    except Exception as e:
        raise IOError(f"Cannot open file: {e}")

    if verbose:
        print(f"\n Checking file: {h5_path}")
        print("=" * 80)

    # --- 1. Basic structure ---
    required_groups = [
        "allImgs",
        "trial_imgids",
        "rois",
        "prfSampleLev",
        "prfSampleLevOri",
        "prfParam",
    ]

    for g in required_groups:
        if g not in f:
            issues.append(f"Missing group or dataset: {g}")
            if verbose:
                print(f" Missing: {g}")

    # --- 2. allImgs vs trial_imgids consistency ---
    if "allImgs" in f and "trial_imgids" in f:
        allImgs = f["allImgs"][:]
        trial_imgids = f["trial_imgids"][:]
        uniq_trials = np.unique(trial_imgids)
        if not np.array_equal(np.sort(allImgs), np.sort(uniq_trials)):
            issues.append("Mismatch: allImgs ≠ unique(trial_imgids)")
            if verbose:
                print(f" allImgs mismatch: {len(allImgs)} vs {len(uniq_trials)} unique trial IDs")
        else:
            if verbose:
                print(f" allImgs consistent ({len(allImgs)} images)")

    # --- 3. ROI dataset consistency ---
    if "rois" in f:
        rois = f["rois"][:]
        if verbose:
            print(f"ROIs listed: {rois.tolist()}")

        for roinum in rois:
            lev_key = f"prfSampleLev/roi_{roinum}"
            ori_key = f"prfSampleLevOri/roi_{roinum}"

            if lev_key not in f or ori_key not in f:
                issues.append(f"Missing dataset for ROI {roinum}")
                continue

            lev = f[lev_key]
            ori = f[ori_key]

            # check shape alignment
            if lev.shape[0] != ori.shape[0]:
                issues.append(f"Image count mismatch: {lev_key} vs {ori_key}")
            if lev.shape[1] != ori.shape[1]:
                issues.append(f"Voxel count mismatch: {lev_key} vs {ori_key}")

            if verbose:
                print(f"ROI {roinum}: prfLev {lev.shape}, prfLevOri {ori.shape}")

            # optional data checks
            if check_values:
                try:
                    sub = lev[:: max(1, len(lev)//10)]  # sample ~10%
                    mean_val = np.nanmean(sub)
                    std_val = np.nanstd(sub)
                    if np.isnan(mean_val) or np.isinf(mean_val):
                        issues.append(f"NaN/Inf values in {lev_key}")
                    if abs(mean_val) < atol and std_val < atol:
                        issues.append(f"All zeros or constant in {lev_key}")
                except Exception as e:
                    issues.append(f"Corrupt data in {lev_key}: {e}")

    # --- 4. prfParam consistency ---
    if "prfParam" in f:
        params = list(f["prfParam"].keys())
        if verbose:
            print(f"Parameter subgroups: {params}")

        # detect duplicate param keys (possible corruption)
        seen = set()
        for p in params:
            if p in seen:
                issues.append(f"Duplicate param subgroup: {p}")
            seen.add(p)

        # check per-ROI parameter length consistency
        for param in ["ecc", "ang", "sz", "x", "y", "r2"]:
            if param not in f["prfParam"]:
                issues.append(f"Missing parameter {param}")
                continue

            roi_grp = f["prfParam"][param]
            for roi_name, dset in roi_grp.items():
                vals = dset[:]
                if np.any(np.isnan(vals)):
                    issues.append(f"NaN in {param}/{roi_name}")
                if np.any(np.isinf(vals)):
                    issues.append(f"Inf in {param}/{roi_name}")
                if np.all(np.abs(vals) < atol):
                    issues.append(f"{param}/{roi_name} all zeros")

    # --- 5. Attribute sanity checks ---
    attrs = f.attrs
    required_attrs = ["numLevels", "numOrientations", "pixPerDeg", "roi_names"]
    for a in required_attrs:
        if a not in attrs:
            issues.append(f"Missing attribute: {a}")
        elif verbose:
            print(f"{a}: {attrs[a]}")

    # --- 6. Duplicate dataset detection (generic) ---
    all_keys = []
    f.visit(all_keys.append)
    if len(all_keys) != len(set(all_keys)):
        issues.append("Duplicate dataset paths found")

    # --- 7. Quick summary ---
    if verbose:
        print("\nSummary:")
        if issues:
            for i in issues:
                print(" ", i)
            print(f"Total issues found: {len(issues)}")
        else:
            print("No problems detected — structure and data look consistent.")

    f.close()
    return issues


# --- Example usage ---
h5_path = "/content/drive/MyDrive/V1_Drift/prfsample_expand/prfSampleStim_v4_sub5.h5"
issues = check_prf_hdf5_integrity(h5_path)
if issues:
   print("\n File has potential issues — see list above.")
else:
   print("\n File integrity confirmed clean.")



with h5py.File(h5_path, "r") as f:
    print("pixPerDeg:", f.attrs["pixPerDeg"])


# Visual Sanity Check: pRF-Sampled Feature Responses

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# User Configuration
# ============================================================

BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/prfsample_expand"

SUBJECT = 4
VISUAL_REGION = 4
ROI_ID = 6

USE_ORI = False

VOXEL_INDEX = 100
IMAGE_INDEX = 0

SUMMARY_METRIC = "mean"  # "mean", "norm", "max"

h5_path = f"{BASE_FOLDER}/prfSampleStim_v{VISUAL_REGION}_sub{SUBJECT}.h5"


# ============================================================
# Load data
# ============================================================

with h5py.File(h5_path, "r") as f:
    allImgs = f["allImgs"][:]

    if USE_ORI:
        data = f[f"prfSampleLevOri/roi_{ROI_ID}"][:]
        data = data.reshape(data.shape[0], data.shape[1], -1)
    else:
        data = f[f"prfSampleLev/roi_{ROI_ID}"][:]

print("Loaded:", h5_path)
print("Shape:", data.shape)


# ============================================================
# Helper
# ============================================================

def summarize(x, metric="mean"):
    if metric == "mean":
        return np.mean(x, axis=-1)
    elif metric == "norm":
        return np.linalg.norm(x, axis=-1)
    elif metric == "max":
        return np.max(x, axis=-1)


# ============================================================
# Histogram 1: One voxel across all images
# ============================================================

vals_voxel = summarize(data[:, VOXEL_INDEX, :], SUMMARY_METRIC)

plt.figure(figsize=(7, 4))
plt.hist(vals_voxel, bins=50)
plt.title(f"Voxel {VOXEL_INDEX}: responses across all images")
plt.xlabel(f"{SUMMARY_METRIC} feature value")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()


# ============================================================
# Histogram 2: One image across all voxels
# ============================================================

vals_image = summarize(data[IMAGE_INDEX, :, :], SUMMARY_METRIC)

plt.figure(figsize=(7, 4))
plt.hist(vals_image, bins=50)
plt.title(f"Image {allImgs[IMAGE_INDEX]}: responses across all voxels")
plt.xlabel(f"{SUMMARY_METRIC} feature value")
plt.ylabel("Number of voxels")
plt.tight_layout()
plt.show()

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

PRFSAMPLE_DIR = "/content/drive/MyDrive/V1_Drift/prfsample_expand"

SAVE_FIGS = True
FIG_DIR = "/content/drive/MyDrive/V1_Drift/plots/prf_sampling_checks"
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DPI = 300

VREG_NAMES = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}

ROI_NAMES = {
    0: "V1v", 1: "V1d",
    2: "V2v", 3: "V3d",
    4: "V3v", 5: "V3d",
    6: "hV4",
}

# =========================================================
# PLOT STYLE FOR POWERPOINT
# =========================================================
FIG_WIDE = (8, 6)
FIG_SQUARE = (8, 8)

TITLE_FS = 18
LABEL_FS = 16
TICK_FS = 13
LEGEND_FS = 12
CBAR_FS = 12


def style_axis(ax, xlabel=None, ylabel=None, title=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=LABEL_FS, fontweight="bold")
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=LABEL_FS, fontweight="bold")
    if title is not None:
        ax.set_title(title, fontsize=TITLE_FS, fontweight="bold", pad=14, wrap=True)
    ax.tick_params(axis="both", labelsize=TICK_FS)


def style_colorbar(cbar, label):
    cbar.set_label(label, fontsize=CBAR_FS, fontweight="bold")
    cbar.ax.tick_params(labelsize=TICK_FS)


def save_current_fig(subject, visual_region, roi_name, fig_name):
    if not SAVE_FIGS:
        return

    safe_roi = roi_name.replace("/", "_")
    out_dir = os.path.join(
        FIG_DIR,
        f"sub{subject:02d}",
        f"v{visual_region}_{VREG_NAMES.get(visual_region, 'ROI')}",
        safe_roi,
    )
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, f"{fig_name}.png")
    plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    print(f"Saved: {out_path}")


def get_prf_sampling_bounds_from_file(f):
    interp_img_size = int(f.attrs["interpImgSize"]) if "interpImgSize" in f.attrs else 256
    pix_per_deg = float(f.attrs["pixPerDeg"]) if "pixPerDeg" in f.attrs else interp_img_size / 8.4
    half = interp_img_size / 2.0

    return {
        "unit_name": "pixels",
        "stim_half_size": half,
        "screen_half_size": half,
        "plot_lim": half * 1.10,
        "pix_per_deg": pix_per_deg,
        "interp_img_size": interp_img_size,
    }


def add_sampling_bounds(ax, bounds):
    stim_half = bounds["stim_half_size"]
    screen_half = bounds["screen_half_size"]

    canvas_rect = Rectangle(
        (-screen_half, -screen_half),
        2 * screen_half,
        2 * screen_half,
        linewidth=2,
        edgecolor="black",
        linestyle="--",
        facecolor="none",
        label="model grid / sampling canvas",
    )
    ax.add_patch(canvas_rect)

    stim_rect = Rectangle(
        (-stim_half, -stim_half),
        2 * stim_half,
        2 * stim_half,
        linewidth=2.5,
        edgecolor="red",
        facecolor="none",
        label="stimulus image extent",
    )
    ax.add_patch(stim_rect)


def plot_prf_map_with_bounds(
    x,
    y,
    sz_deg,
    subject,
    visual_region,
    roi_name,
    max_voxels_to_plot=3000,
    bounds=None,
):
    if bounds is None:
        raise ValueError("bounds must be provided from get_prf_sampling_bounds_from_file(f).")

    n_vox = len(x)

    stim_half = bounds["stim_half_size"]
    plot_lim = bounds["plot_lim"]
    pix_per_deg = bounds["pix_per_deg"]

    sz_pix = sz_deg * pix_per_deg

    inside_stim = (
        (x >= -stim_half) & (x <= stim_half) &
        (y >= -stim_half) & (y <= stim_half)
    )

    print("\nCoordinate system from pRF sampling file/code: pixels")
    print(f"interpImgSize: {bounds['interp_img_size']}")
    print(f"pixPerDeg: {pix_per_deg:.3f}")
    print(f"Stimulus/model-grid half-size: +/-{stim_half:.1f} pixels")
    print(
        f"Voxels with pRF center inside stimulus/model bounds: "
        f"{inside_stim.sum()} / {n_vox} ({100 * inside_stim.mean():.1f}%)"
    )
    print(
        f"pRF size range: "
        f"{np.nanmin(sz_deg):.3f}-{np.nanmax(sz_deg):.3f} deg | "
        f"{np.nanmin(sz_pix):.3f}-{np.nanmax(sz_pix):.3f} pixels"
    )

    plot_idx = np.arange(n_vox)
    if n_vox > max_voxels_to_plot:
        rng = np.random.default_rng(0)
        plot_idx = rng.choice(n_vox, size=max_voxels_to_plot, replace=False)

    fig, ax = plt.subplots(figsize=FIG_SQUARE)

    add_sampling_bounds(ax, bounds)

    for idx in plot_idx:
        if (
            np.isfinite(x[idx]) and
            np.isfinite(y[idx]) and
            np.isfinite(sz_pix[idx]) and
            sz_pix[idx] > 0
        ):
            circle = Circle(
                (x[idx], y[idx]),
                radius=sz_pix[idx],
                fill=False,
                linewidth=0.6,
                alpha=0.12,
            )
            ax.add_patch(circle)

    ax.scatter(x[plot_idx], y[plot_idx], s=10, alpha=0.45, label="pRF centers")

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlim(-plot_lim, plot_lim)
    ax.set_ylim(-plot_lim, plot_lim)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right", fontsize=LEGEND_FS)

    style_axis(
        ax,
        xlabel="pRF x position (pixels)",
        ylabel="pRF y position (pixels)",
        title=f"Subject {subject} | {roi_name}: pRF centers and sizes",
    )

    plt.tight_layout()
    save_current_fig(subject, visual_region, roi_name, "01_prf_centers_with_size_circles")
    plt.show()

    return bounds, sz_pix


def plot_prf_centers_only(
    x,
    y,
    subject,
    visual_region,
    roi_name,
    bounds,
    color_by_eccentricity=False,
):
    plot_lim = bounds["plot_lim"]
    pix_per_deg = bounds["pix_per_deg"]

    fig, ax = plt.subplots(figsize=FIG_SQUARE, constrained_layout=True)

    add_sampling_bounds(ax, bounds)

    if color_by_eccentricity:
        ecc_deg = np.sqrt(x**2 + y**2) / pix_per_deg

        sc = ax.scatter(x, y, c=ecc_deg, s=18, alpha=0.75)
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        style_colorbar(cbar, "pRF eccentricity (deg)")
        fig_name = "03_prf_centers_colored_by_eccentricity"
        title = f"Subject {subject} | {roi_name}: pRF centers\ncolored by eccentricity"
    else:
        ax.scatter(x, y, s=18, alpha=0.65, label="pRF centers")
        ax.legend(loc="upper right", fontsize=LEGEND_FS)
        fig_name = "02_prf_centers_only"
        title = f"Subject {subject} | {roi_name}: pRF centers"

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_xlim(-plot_lim, plot_lim)
    ax.set_ylim(-plot_lim, plot_lim)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

    style_axis(
        ax,
        xlabel="pRF x position (pixels)",
        ylabel="pRF y position (pixels)",
        title=title,
    )

    ax.title.set_wrap(True)

    save_current_fig(subject, visual_region, roi_name, fig_name)
    plt.show()


def plot_prf_density_with_bounds(
    x,
    y,
    subject,
    visual_region,
    roi_name,
    bounds=None,
    bins=40,
):
    if bounds is None:
        raise ValueError("bounds must be provided from get_prf_sampling_bounds_from_file(f).")

    plot_lim = bounds["plot_lim"]

    fig, ax = plt.subplots(figsize=FIG_SQUARE, constrained_layout=True)

    h = ax.hist2d(
        x,
        y,
        bins=bins,
        range=[[-plot_lim, plot_lim], [-plot_lim, plot_lim]],
    )

    cbar = fig.colorbar(h[3], ax=ax, fraction=0.046, pad=0.04)
    style_colorbar(cbar, "Number of voxels")

    add_sampling_bounds(ax, bounds)

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_xlim(-plot_lim, plot_lim)
    ax.set_ylim(-plot_lim, plot_lim)
    ax.set_aspect("equal")

    ax.legend(loc="upper right", fontsize=LEGEND_FS)

    style_axis(
        ax,
        xlabel="pRF x position (pixels)",
        ylabel="pRF y position (pixels)",
        title=f"Subject {subject} | {roi_name}: voxel count\nby pRF location",
    )

    ax.title.set_wrap(True)

    save_current_fig(subject, visual_region, roi_name, "04_prf_density_map")
    plt.show()

def inspect_prf_sampling(subject=1, visual_region=1, max_voxels_to_plot=3000):
    prf_file = os.path.join(
        PRFSAMPLE_DIR,
        f"prfSampleStim_v{visual_region}_sub{subject}.h5",
    )

    if not os.path.exists(prf_file):
        raise FileNotFoundError(f"Missing file:\n{prf_file}")

    print(f"Reading: {prf_file}")

    with h5py.File(prf_file, "r") as f:
        print("\nTop-level keys:")
        print(list(f.keys()))

        num_levels = int(f.attrs["numLevels"])
        num_orientations = int(f.attrs["numOrientations"])
        bounds = get_prf_sampling_bounds_from_file(f)

        print(f"\nnumLevels: {num_levels}")
        print(f"numOrientations: {num_orientations}")
        print(f"interpImgSize: {bounds['interp_img_size']}")
        print(f"pixPerDeg: {bounds['pix_per_deg']:.3f}")

        roi_keys = sorted(list(f["prfSampleLev"].keys()))
        print("\nROIs in file:", roi_keys)

        for roi_key in roi_keys:
            roinum = int(roi_key.replace("roi_", ""))
            roi_name = ROI_NAMES.get(roinum, f"roi_{roinum}")

            print("\n" + "=" * 70)
            print(
                f"Subject {subject}, visual region {visual_region} "
                f"({VREG_NAMES.get(visual_region)}), ROI {roinum} ({roi_name})"
            )

            prf_lev = f[f"prfSampleLev/{roi_key}"]
            prf_ori = f[f"prfSampleLevOri/{roi_key}"]

            print("prfSampleLev shape:   ", prf_lev.shape)
            print("prfSampleLevOri shape:", prf_ori.shape)

            if "prfParam" not in f:
                print("No prfParam group found. Skipping pRF plots.")
                continue

            x = f[f"prfParam/x/{roi_key}"][:]
            y = f[f"prfParam/y/{roi_key}"][:]
            sz_deg = f[f"prfParam/sz/{roi_key}"][:]

            ecc_path = f"prfParam/ecc/{roi_key}"
            r2_path = f"prfParam/r2/{roi_key}"

            ecc_deg = (
                f[ecc_path][:]
                if ecc_path in f
                else np.sqrt(x**2 + y**2) / bounds["pix_per_deg"]
            )
            r2 = f[r2_path][:] if r2_path in f else None

            x_deg = x / bounds["pix_per_deg"]
            y_deg = y / bounds["pix_per_deg"]

            print(f"Number of voxels: {len(x)}")
            print(f"x range:       {np.nanmin(x):.2f} to {np.nanmax(x):.2f} pixels")
            print(f"y range:       {np.nanmin(y):.2f} to {np.nanmax(y):.2f} pixels")
            print(f"x range:       {np.nanmin(x_deg):.2f} to {np.nanmax(x_deg):.2f} deg")
            print(f"y range:       {np.nanmin(y_deg):.2f} to {np.nanmax(y_deg):.2f} deg")
            print(f"size range:    {np.nanmin(sz_deg):.2f} to {np.nanmax(sz_deg):.2f} deg")
            print(f"ecc range:     {np.nanmin(ecc_deg):.2f} to {np.nanmax(ecc_deg):.2f} deg")
            if r2 is not None:
                print(f"pRF R2 range:  {np.nanmin(r2):.2f} to {np.nanmax(r2):.2f}")

            bounds, sz_pix = plot_prf_map_with_bounds(
                x=x,
                y=y,
                sz_deg=sz_deg,
                subject=subject,
                visual_region=visual_region,
                roi_name=roi_name,
                max_voxels_to_plot=max_voxels_to_plot,
                bounds=bounds,
            )

            plot_prf_centers_only(
                x=x,
                y=y,
                subject=subject,
                visual_region=visual_region,
                roi_name=roi_name,
                bounds=bounds,
                color_by_eccentricity=False,
            )

            plot_prf_centers_only(
                x=x,
                y=y,
                subject=subject,
                visual_region=visual_region,
                roi_name=roi_name,
                bounds=bounds,
                color_by_eccentricity=True,
            )

            plot_prf_density_with_bounds(
                x=x,
                y=y,
                subject=subject,
                visual_region=visual_region,
                roi_name=roi_name,
                bounds=bounds,
                bins=40,
            )

            # pRF size distribution in degrees
            fig, ax = plt.subplots(figsize=FIG_WIDE)
            ax.hist(sz_deg, bins=30)
            style_axis(
                ax,
                xlabel="pRF size (degrees)",
                ylabel="Number of voxels",
                title=f"Subject {subject} | {roi_name}: pRF size distribution in degrees",
            )
            plt.tight_layout()
            save_current_fig(subject, visual_region, roi_name, "05_prf_size_distribution_degrees")
            plt.show()

            # pRF size distribution in pixels
            fig, ax = plt.subplots(figsize=FIG_WIDE)
            ax.hist(sz_pix, bins=30)
            style_axis(
                ax,
                xlabel="pRF size (pixels)",
                ylabel="Number of voxels",
                title=f"Subject {subject} | {roi_name}: pRF size distribution in pixels",
            )
            plt.tight_layout()
            save_current_fig(subject, visual_region, roi_name, "06_prf_size_distribution_pixels")
            plt.show()

            # Mean energy per level
            lev_data = prf_lev[:]
            mean_level_energy = np.nanmean(lev_data, axis=(0, 1))
            sem_level_energy = np.nanstd(lev_data, axis=(0, 1)) / np.sqrt(
                np.prod(lev_data.shape[:2])
            )

            level_labels = [f"L{i+1}" for i in range(num_levels)] + ["Low SF", "High SF"]

            fig, ax = plt.subplots(figsize=FIG_WIDE)
            ax.bar(
                np.arange(len(mean_level_energy)),
                mean_level_energy,
                yerr=sem_level_energy,
            )
            ax.set_xticks(np.arange(len(mean_level_energy)))
            ax.set_xticklabels(level_labels, fontsize=TICK_FS)
            style_axis(
                ax,
                xlabel="Spatial-frequency level",
                ylabel="Mean sampled energy",
                title=f"Subject {subject} | {roi_name}: mean energy by level",
            )
            plt.tight_layout()
            save_current_fig(subject, visual_region, roi_name, "07_mean_energy_by_level")
            plt.show()

            # Mean energy per orientation
            ori_data = prf_ori[:]
            mean_ori_energy = np.nanmean(ori_data, axis=(0, 1, 2))
            sem_ori_energy = np.nanstd(ori_data, axis=(0, 1, 2)) / np.sqrt(
                ori_data.shape[0] * ori_data.shape[1] * ori_data.shape[2]
            )

            fig, ax = plt.subplots(figsize=FIG_WIDE)
            ax.bar(
                np.arange(num_orientations),
                mean_ori_energy,
                yerr=sem_ori_energy,
            )
            ax.set_xticks(np.arange(num_orientations))
            ax.set_xticklabels([f"O{i+1}" for i in range(num_orientations)], fontsize=TICK_FS)
            style_axis(
                ax,
                xlabel="Orientation",
                ylabel="Mean sampled energy",
                title=f"Subject {subject} | {roi_name}: mean energy by orientation",
            )
            plt.tight_layout()
            save_current_fig(subject, visual_region, roi_name, "08_mean_energy_by_orientation")
            plt.show()

            # Mean energy per level × orientation
            mean_level_ori = np.nanmean(ori_data, axis=(0, 1))

            print("\nMean energy per level × orientation:")
            print(mean_level_ori)

            for lev in range(num_levels):
                vals = mean_level_ori[lev]
                min_val = np.nanmin(vals)
                max_val = np.nanmax(vals)
                ratio = max_val / min_val if min_val > 0 else np.nan

                print(f"\nLevel {lev + 1}")
                for ori in range(num_orientations):
                    print(f"  O{ori + 1}: {vals[ori]:.3f}")
                print(f"  max/min ratio: {ratio:.3f}")

            fig, ax = plt.subplots(figsize=FIG_WIDE)
            im = ax.imshow(mean_level_ori, aspect="auto")
            cbar = plt.colorbar(im, ax=ax)
            style_colorbar(cbar, "Mean sampled energy")

            ax.set_xticks(np.arange(num_orientations))
            ax.set_xticklabels([f"O{i+1}" for i in range(num_orientations)], fontsize=TICK_FS)

            ax.set_yticks(np.arange(num_levels))
            ax.set_yticklabels([f"L{i+1}" for i in range(num_levels)], fontsize=TICK_FS)

            style_axis(
                ax,
                xlabel="Orientation",
                ylabel="Level",
                title=f"Subject {subject} | {roi_name}: level × orientation energy",
            )

            plt.tight_layout()
            save_current_fig(subject, visual_region, roi_name, "09_level_by_orientation_energy")
            plt.show()

            del lev_data, ori_data


for subject in range(1, 9):
    for visual_region in range(1, 5):
        try:
            print("\n" + "#" * 90)
            print(f"Running subject {subject}, visual region {visual_region}")
            print("#" * 90)

            inspect_prf_sampling(
                subject=subject,
                visual_region=visual_region,
                max_voxels_to_plot=3000
            )

        except FileNotFoundError as e:
            print(f"Skipping missing file: subject {subject}, visual region {visual_region}")
            print(e)

# check one pyramid file

In [ ]:
import os, numpy as np

# ---- config ----
# Inspect a single file: set img_ids = [0]  (this loads pyrImg0.npz)
# Inspect several:       set img_ids = [0, 2, 3, 6, 11]  (your earlier [1,3,4,7,12] are 1-based → minus 1)
img_ids = [0]   # change this list as needed
root = "/content/drive/MyDrive/V1_Drift/NSD/pyramid_expand"
atol = 1e-5     # tolerance for lev vs sum over orientations

def unwrap_modelOri(modelOri):
    """Handle various container formats and return a Python list of length L(+2),
    where each item for band levels is an array of shape (O,H,W) or a list of O arrays (H,W)."""
    # Some exporters pack as a 1-element object ndarray containing a Python list
    if isinstance(modelOri, np.ndarray) and modelOri.shape == (1,) and isinstance(modelOri[0], list):
        modelOri = modelOri[0]
    return modelOri

def summarize_map(name, arr):
    arr = np.asarray(arr)
    nz = int(np.count_nonzero(arr))
    has_nan = bool(np.isnan(arr).any())
    has_inf = bool(np.isinf(arr).any())
    print(f"    {name}: shape={arr.shape} dtype={arr.dtype} "
          f"min={arr.min():.6g} max={arr.max():.6g} mean={arr.mean():.6g} nz={nz} "
          f"NaN={has_nan} Inf={has_inf}")

def check_pyr_file(img_id):
    path = os.path.join(root, f"pyrImg{img_id}.npz")
    print(f"\n=== pyr file: {path} ===")
    if not os.path.exists(path):
        print("  MISSING")
        return False

    with np.load(path, allow_pickle=True) as data:
        sumOri = data["sumOri"]
        modelOri = unwrap_modelOri(data["modelOri"])

    # Basic structure
    print(f"  sumOri: {len(sumOri)} levels")
    for i, lev in enumerate(sumOri):
        summarize_map(f"sumOri[{i}]", lev)

    # modelOri structure
    if isinstance(modelOri, np.ndarray) and modelOri.dtype != object:
        # Sometimes modelOri is a pure ndarray with shape (L, O, H, W) or (L, ?, H, W)
        print(f"\n  modelOri: ndarray with shape {modelOri.shape} (dtype={modelOri.dtype})")
        as_list = []
        for i in range(modelOri.shape[0]):
            as_list.append(modelOri[i])
        modelOri = as_list
    else:
        print(f"\n  modelOri: {len(modelOri)} levels (list-like)")

    # Per-level details
    Lp2 = len(sumOri)                 # should be numLevels+2
    L   = None
    if len(modelOri) in (Lp2, Lp2-2):
        # Some dumps include entries for residuals (low/high); others only bands
        L = Lp2 - 2
    else:
        # fallback: assume orientations exist for all min(len(sumOri), len(modelOri)) levels
        L = min(len(sumOri), len(modelOri))

    # Inspect band levels (0..L-1)
    for i in range(L):
        mi = modelOri[i]
        if isinstance(mi, list):
            O = len(mi)
            shapes = set([np.asarray(x).shape for x in mi])
            print(f"  modelOri[{i}]: list of {O} orientations; unique shapes={list(shapes)}")
            # Summaries for first two orientations for brevity
            for j in range(min(O, 2)):
                summarize_map(f"modelOri[{i}][{j}]", mi[j])
            # Consistency: sum over orientations ≈ sumOri[i]
            stacked = np.stack([np.asarray(x) for x in mi], axis=0)  # (O,H,W)
        else:
            mi = np.asarray(mi)
            assert mi.ndim == 3, f"modelOri[{i}] ndim={mi.ndim}, expected 3 (O,H,W)"
            O = mi.shape[0]
            print(f"  modelOri[{i}]: ndarray (O,H,W) = {mi.shape}")
            for j in range(min(O, 2)):
                summarize_map(f"modelOri[{i}][{j}]", mi[j])
            stacked = mi  # (O,H,W)

        # Check dimension match with sumOri
        so = np.asarray(sumOri[i])
        if so.shape != stacked.shape[1:]:
            print(f"    [WARN] shape mismatch: sumOri[{i}] {so.shape} vs modelOri[{i}] {stacked.shape[1:]}")
        # Lev vs sum over orientations
        ori_sum = stacked.sum(axis=0)
        max_abs = float(np.max(np.abs(so - ori_sum)))
        print(f"    check: ||sumOri[{i}] - sum(modelOri[{i}], ori)||_∞ = {max_abs:.6g} "
              f"({'OK' if max_abs <= atol else '> tol'})")

    # Residuals (low/high) if present
    if Lp2 >= 2:
        for i in (Lp2-2, Lp2-1):
            summarize_map(f"sumOri residual[{i}]", sumOri[i])
        if len(modelOri) > L:
            print(f"  Note: modelOri has {len(modelOri)} entries; ignoring residual orientation entries at indices >= {L}.")

    return True

# ---- run ----
ok_all = True
for iid in img_ids:
    ok_all &= check_pyr_file(iid)

print("\nDone. OK =", ok_all)
